# Day 27: Titanic Pipeline Architecture with Production Logging
### Target: Build a robust, logged, and reusable classification pipeline on the Titanic dataset.

In this notebook, we set up a production-grade machine learning preprocessing and training pipeline. We use scikit-learn's `Pipeline` and `ColumnTransformer` to handle imputation and encoding, and instrument the entire workflow using Python's native `logging` module.

In [2]:
import os
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
import joblib

## 1. Configure Logging
We configure a logger that writes logs to both a file (`pipeline.log`) and the standard output console.

In [3]:
# Setup logging configuration
logger = logging.getLogger('TitanicPipeline')
logger.setLevel(logging.INFO)

# Avoid adding duplicate handlers if already configured
if not logger.handlers:
    # File handler
    fh = logging.FileHandler('pipeline.log')
    fh.setLevel(logging.INFO)
    
    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    
    # Formatter
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    
    logger.addHandler(fh)
    logger.addHandler(ch)

logger.info('Logging successfully initialized.')

2026-06-29 20:27:04,597 - TitanicPipeline - INFO - Logging successfully initialized.


## 2. Load the Dataset
We load the Titanic training dataset from the workspace.

In [4]:
data_path = 'train.csv'
logger.info(f'Loading dataset from path: {data_path}')

if not os.path.exists(data_path):
    logger.error(f'Dataset file not found at {data_path}!')
    raise FileNotFoundError(f'File not found: {data_path}')

df = pd.read_csv(data_path)
logger.info(f'Dataset loaded successfully with shape: {df.shape}')
df.head()

2026-06-29 20:27:08,768 - TitanicPipeline - INFO - Loading dataset from path: train.csv
2026-06-29 20:27:08,810 - TitanicPipeline - INFO - Dataset loaded successfully with shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Train-Test Split
We split our features ($X$) and target ($y$) variables before preprocessing to prevent data leakage.

In [5]:
target = 'Survived'
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = df[features]
y = df[target]

logger.info(f'Features selected: {features}')
logger.info(f'Splitting data into train and validation sets (test_size=0.2)')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
logger.info(f'Training size: {X_train.shape}, Validation size: {X_val.shape}')

2026-06-29 20:27:12,678 - TitanicPipeline - INFO - Features selected: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
2026-06-29 20:27:12,679 - TitanicPipeline - INFO - Splitting data into train and validation sets (test_size=0.2)
2026-06-29 20:27:12,699 - TitanicPipeline - INFO - Training size: (712, 7), Validation size: (179, 7)


## 4. Define Preprocessing & Pipeline Architecture
We define numerical and categorical pipelines, join them in a `ColumnTransformer`, and bind them with a `LogisticRegression` model.

In [6]:
logger.info('Defining preprocessing sub-pipelines.')

numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_cols = ['Pclass', 'Sex', 'Embarked']

# Numerical Pipeline: Impute with median, scale with standard scaling
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute with mode, one-hot encode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

logger.info('Assembling ColumnTransformer.')
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_cols),
    ('cat', cat_transformer, categorical_cols)
])

logger.info('Building full pipeline with Logistic Regression model.')
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

2026-06-29 20:27:14,555 - TitanicPipeline - INFO - Defining preprocessing sub-pipelines.
2026-06-29 20:27:14,556 - TitanicPipeline - INFO - Assembling ColumnTransformer.
2026-06-29 20:27:14,566 - TitanicPipeline - INFO - Building full pipeline with Logistic Regression model.


## 5. Fit the Pipeline
We train the model by calling `.fit` on the pipeline. This executes all preprocessing steps safely on the training data.

In [7]:
logger.info('Fitting model pipeline on training data.')
model_pipeline.fit(X_train, y_train)
logger.info('Model pipeline fitting complete.')

2026-06-29 20:27:16,213 - TitanicPipeline - INFO - Fitting model pipeline on training data.
2026-06-29 20:27:16,269 - TitanicPipeline - INFO - Model pipeline fitting complete.


## 6. Validate & Predict
We evaluate our model on the holdout validation set.

In [8]:
logger.info('Running prediction inference on validation set.')
y_pred = model_pipeline.predict(X_val)
accuracy = (y_pred == y_val).mean()

logger.info(f'Validation set evaluation completed. Accuracy: {accuracy:.4f}')
print(f'Validation Accuracy: {accuracy:.4f}')

2026-06-29 20:27:17,257 - TitanicPipeline - INFO - Running prediction inference on validation set.
2026-06-29 20:27:17,288 - TitanicPipeline - INFO - Validation set evaluation completed. Accuracy: 0.8045


Validation Accuracy: 0.8045


## 7. Serialize the Pipeline
We save our complete trained pipeline to a joblib file for downstream deployment or verification.

In [9]:
model_filename = 'titanic_pipeline.joblib'
logger.info(f'Saving trained pipeline model to file: {model_filename}')
joblib.dump(model_pipeline, model_filename)
logger.info('Model serialization successful!')

2026-06-29 20:27:18,297 - TitanicPipeline - INFO - Saving trained pipeline model to file: titanic_pipeline.joblib
2026-06-29 20:27:18,305 - TitanicPipeline - INFO - Model serialization successful!
